In [0]:
from pyspark.sql.functions import col,initcap,lit

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run /Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/00.Configurations

In [0]:
%run "/Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/002.silver helper functions"

In [0]:
bronze_table = f'{catalog}.{bronze_schema}.constructors'
silver_table = f'{catalog}.{silver_schema}.constructors'

In [0]:
constructors_df = (
                   spark.table(bronze_table)
                        .filter(col("batch_id") == lit(v_batch_id))
                        .drop(col("url"))
                        .withColumnsRenamed(
                            {"constructorId":"constructor_id",
                             "name":"constructor_name"})
                        .dropDuplicates(["constructor_id"])
                        .withColumn("nationality",initcap(col("nationality")))
)

In [0]:
write_to_silver(
    input_df=constructors_df,
    table_name=silver_table,
    merge_condition=col("t.constructor_id") == col("s.constructor_id"),
    columns_to_update= ["constructor_id","constructor_name","nationality","ingestion_time","filename"]
)